### 1、导包

In [1]:
import math
from pickletools import optimize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass

torch.manual_seed(2048)

### 2、定义 GPT 参数

In [2]:
@dataclass
class GPTConfig:
    block_size: int = 512 # 文本的最大长度 (max sequence length)
    batch_size: int = 12
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768 # hidden dimension，token embedding 后的维度
    hidden_dim: int = n_embd
    drop_out: float = 0.1
    head_size: int = n_embd // n_head # 如果只有单头注意力，那么可有可无，但下面打算构建多头注意力，因此将维度分为 n_head 份
    vocab_size: int = 50257

### 3、定义模型

In [10]:
# single head attention
class SingleHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.key = nn.Linear(config.hidden_dim, config.head_size) # (B, T, C) -> (B, T, H)
        self.value = nn.Linear(config.hidden_dim, config.head_size)
        self.query = nn.Linear(config.hidden_dim, config.head_size)

        # attention mask 通过 register_buffer 方法注册一个 buffer，这个 buffer 不需要梯度
        self.register_buffer(
            "attention_mask",
            torch.tril(torch.ones(config.block_size, config.block_size)) # block size 即文本的最大长度
        )
        self.drop_out = nn.Dropout(config.drop_out)

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.size()
        k = self.key(x)
        v = self.value(x)
        q = self.query(x)
        weight = q @ k.transpose(-2, -1) # 该行与下一行对应 MatMul(Q, K^T)
        weight = weight.masked_fill(self.attention_mask[:seq_len, :seq_len] == 0, float("-inf")) / math.sqrt(self.head_size)
        weight = F.softmax(weight, dim=-1) # 过 softmax
        weight = self.drop_out(weight) # dropout来防止过拟合
        output = weight @ v # MatMul(softmax(QK^T), V)，算出 attention 的输出
        return output

# multi head attention
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList( # 多个单头注意力拼接成多头注意力
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.drop_out = nn.Dropout(config.drop_out)

    def forward(self, x):
        output = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )
        output = self.proj(output) # 将多头注意力的输出进行线性变换，映射到原始的维度
        output = self.drop_out(output)
        return output

# feed forward (MLP layer)
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.hidden_dim, 4 * config.hidden_dim), # 将 hidden dim 扩为 4 倍（扩展前序反馈）
            nn.GELU(),
            nn.Linear(4 * config.hidden_dim, config.hidden_dim),
            nn.Dropout(config.drop_out)
        )
    def forward(self, x):
        return self.net(x)

# block
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.att = MultiHeadAttention(config)
        self.ffn = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.hidden_dim)
        self.ln2 = nn.LayerNorm(config.hidden_dim)
    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# GPT
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        # embedding, position, norm, mlp, block
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )
        self.ln_final = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # 目前的 SLM 模型，会使用 tie weight 来减少参数
        self.token_embedding_table.weight = self.lm_head.weight ### very important

    def _init_weight(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02) # 初始化为正态分布
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None): # idx 输入的是 token id，targets 输入的是目标的 token id，因此 shape 一样
        batch, seq_len = idx.size()
        token_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(seq_len, device=idx.device))
        x = token_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # 如果序列太长，只取最后 block_size 个token
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            # 获取预测
            logits, _ = self(idx_cond)
            # 只关注最后一个时间步的预测
            logits = logits[:, -1, :]  # becomes (B, vocab_size)
            # 应用softmax获取概率
            probs = F.softmax(logits, dim=-1)
            # 采样下一个token
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # 附加到序列上
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

### 4、构建 dataset

In [11]:
class MyDataset(Dataset):
    def __init__(self, path, block_size=512):
        import tiktoken
        self.enc = tiktoken.get_encoding("gpt2")
        self.block_size = block_size

        self.encoded_data = [] # 特殊符号分割不同的训练文本 <|endoftext|>
        self.eos_token = self.enc.encode(
            "<|endoftext|>",
            allowed_special={"<|endoftext|>"} # GPT2 特有的特殊符号
        )[0]

        self.max_lines = 1000
        import json
        raw_data = []
        with open(path, "r") as f:
            for i, line in enumerate(f):
                if i >= self.max_lines:
                    break
                try:
                    text = json.loads(line.strip())["text"]
                    raw_data.append(text)
                except Exception as e:
                    continue

        full_encoded = []
        for text in raw_data:
            encoded_text = self.enc.encode(text)
            full_encoded.extend(encoded_text + [self.eos_token])

        # block size 为 512，因此需要对文本进行切割
        for i in range(0, len(full_encoded), self.block_size):
            chuck = full_encoded[i: i + self.block_size + 1]
            if len(chuck) < self.block_size:
                chuck = chuck + [self.eos_token] * (self.block_size + 1 - len(chuck))
            self.encoded_data.append(chuck)

    def __len__(self):
        return len(self.encoded_data)

    def __getitem__(self, idx):
        chuck = self.encoded_data[idx]
        x = torch.tensor(chuck[:-1], dtype=torch.long)
        y = torch.tensor(chuck[1:], dtype=torch.long)
        return x, y

    def encode(self, text):
        return self.enc.encode(text)

    def decode(self, idx):
        return self.enc.decode(idx)

### 5、模型运行

In [12]:
model = GPT(GPTConfig())
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params / 1e6} M total parameters.")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

124.046592 M total parameters.


### 6、训练模型

In [ ]:
# train data
train_dataset = MyDataset('/root/LLM/mobvoi_seq_monkey_general_open_corpus.jsonl')

# split traindataset to train and val
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [0.9, 0.1])

train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=12, shuffle=False)

In [ ]:
def train(model, optimizer, scheduler, train_loader, val_loader, device):
    model.train()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        logits, loss = model(x, targets=y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
        return total_loss

def eval(model, val_loader, device):
    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            val_loss += loss.item()
    return val_loss

for epoch in range(2):
    train_loss = train(model, optimizer, scheduler, train_loader, val_loader, device)
    val_loss = eval(model, val_loader, device)
    print(f'Epoch: {epoch}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}')

    # 保存模型
    avg_val_loss = val_loss / len(val_loader)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': avg_val_loss,
    }
    # 保存每个epoch的模型
    torch.save(checkpoint, f'checkpoints/model_epoch_{epoch}.pt')